In [7]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set Visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print('All libraries imported successfully')
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

All libraries imported successfully
Pandas version: 2.1.4
Numpy version: 1.23.5
Matplotlib version: 3.8.4
Seaborn version: 0.13.2


In [2]:
# Helper Functions
def analyze_dataframe(df, dataset_name):
    """
    Comprehensive analysis of a dataframe
    """
    print("=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    # Basic Info
    print('\nBASIC INFORMATION:')
    print(f'Shape: {df.shape[0]} rows * {df.shape[1]} columns')

    # Column Types
    print('\nCOLUMN TYPES:')
    print(df.dtypes.value_counts())

    # Missing Values
    print('\nMISSING VALUES:')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing[missing > 0],
        'Percentage': missing_pct[missing > 0]
    }).sort_values('Percentage', ascending = False)

    if len(missing_df) > 0:
        print(missing_df)
    else:
        print('No missing values')

    # Duplicates
    duplicates = df.duplicated().sum()
    print(f'\nDUPLICATE ROWS: {duplicates} ({(duplicates / len(df) * 100):.2f}%)')

    # Numerical columns stats
    numerical_cols = df.select_dtypes(include = [np.number]).columns
    if len(numerical_cols) > 0:
        print('\nNUMERICAL COLUMNS STATISTICS:')
        print(df[numerical_cols].describe().T)

    # Categorical columns
    categorical_cols = df.select_dtypes(include = ['object']).columns
    if len(categorical_cols) > 0:
        print(f'\nCATEGORICAL COLUMNS: {len(categorical_cols)}')
        for col in categorical_cols[:5]:  # Show first 5
            print(f'- {col}: {df[col].nunique()} unique values')

    print('\n' + "=" * 70 + "\n")

    return {
        'shape': df.shape,
        'missing': missing_df,
        'duplicates': duplicates,
        'numerical_cols': list(numerical_cols),
        'categorical_cols': list(categorical_cols)
    }

def plot_missing_values(df, title):
    """
    Visualize missing values
    """
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending = False)

    if len(missing) > 0:
        plt.figure(figsize = (10, 6))
        missing.plot(king = 'barh', color = 'coral')
        plt.xlabel('Number of Missing Values')
        plt.title(f'Missing Values - {title}')
        plt.tight_layout()
        plt.show()
    else:
        pritn(f'No missing values in {title}')

def plot_categorical_distribution(df, column, title, top_n = 10):
    """
    Plot distribution of categorical variable
    """
    value_counts = df[column].value_counts().head(top_n)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (15, 5))

    # Bar plot
    value_counts.plot(kind = 'bar', ax = ax1, color = 'skyblue')
    ax1.set_title(f'{title} - Bar Plot')
    ax1.set_xlabel(column)
    ax1.set_ylabel('Count')
    ax1.tick_params(axis = 'x', rotation = 45)

    # Pie chart
    ax2.pie(value_counts, labels = value_counts.index, autopct = '%1.1f%%', startangle = 90)
    ax2.set_title(f'{title} - Distribution')

    plt.tight_layout()
    plt.show()

def plot_numerical_distribution(df, column, title):
    """
    Plot distribution of numerical variable
    """
    fig, axes = plt.subplots(1, 3, figsize = (18, 5))

    # Histogram
    axes[0].hist(df[column].dropna(), bins = 30, color = 'lightgreen', edgecolor = 'black')
    axes[0].set_title(f'{title} - Histogram')
    axes[0].set_xlabel(column)
    axes[0].set_ylabel('Frequency')

    # Box plot
    axes[1].boxplot(df[column].dropna())
    axes[1].set_title(f'{title} - Box Plot')
    axes[1].set_ylabel(column)

    # KDE plot
    df[column].dropna().plot(kind = 'kde', ax = axes[2], color = 'purple')
    axes[2].set_title(f'{title} - Density Plot')
    axes[2].set_xlabel(column)

    plt.tight_layout()
    plt.show()

print('Helper functions defined')

Helper functions defined


# Dataset 1: Customer Support Tickets

In [3]:
# Load Data
tickets = pd.read_csv('/Users/vishalhooda/Downloads/customer_support_tickets.csv')

# Basic analysis
tickets_info = analyze_dataframe(tickets, 'Cusomter Support Tickets')

# Display samplle data
print('SAMPLE DATA (First 5 rows):')
tickets.head()

DATASET: Cusomter Support Tickets

BASIC INFORMATION:
Shape: 8469 rows * 17 columns

COLUMN TYPES:
object     14
int64       2
float64     1
Name: count, dtype: int64

MISSING VALUES:
                              Missing Count  Percentage
Resolution                             5700      67.304
Time to Resolution                     5700      67.304
Customer Satisfaction Rating           5700      67.304
First Response Time                    2819      33.286

DUPLICATE ROWS: 0 (0.00%)

NUMERICAL COLUMNS STATISTICS:
                                count     mean      std    min      25%  \
Ticket ID                    8469.000 4235.000 2444.934  1.000 2118.000   
Customer Age                 8469.000   44.027   15.296 18.000   31.000   
Customer Satisfaction Rating 2769.000    2.991    1.407  1.000    2.000   

                                  50%      75%      max  
Ticket ID                    4235.000 6352.000 8469.000  
Customer Age                   44.000   57.000   70.000  
Cus

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.000
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.000
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.000


In [4]:
# Tickets: Specific Analysis
print('\nTICKET-SPECIFIC INSIGHTS:\n')

# 1. Ticket categories distribution
print('Ticket Categories:')
print(tickets['Ticket Type'].value_counts())
print()

# 2. Priority distribution
print('Priority Levels:')
print(tickets['Ticket Priority'].value_counts())
print()

# 3. Status distribution
print('Ticket Status:')
print(tickets['Ticket Status'].value_counts())
print()

# 4. Resolution time statistics
if 'Time to Resolution' in tickets.columns:
    print('Resolution Time Statistics (minutes):')
    print(tickets['Time to Resolution'].describe())
    print()

# 5. Satisfication scores
if 'Customer Statisfication Rating' in tickets.columns:
    print('Customer Satisfication Scores:')
    print(tickets['Customer Statisfication Rating'].value_counts().sort_index())
    print(f"Average: {tickets['Customer Statisfication Rating'].mean():.2f}/5.0")
    print()


TICKET-SPECIFIC INSIGHTS:

Ticket Categories:
Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

Priority Levels:
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

Ticket Status:
Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

Resolution Time Statistics (minutes):
count                    2769
unique                   2728
top       2023-06-01 17:14:42
freq                        3
Name: Time to Resolution, dtype: object

